In [1]:
import os
import marimo as mo
import deepcor

os.chdir(mo.notebook_dir())  # all paths relative to the notebook

# DeepCor — simple training (CVAE v1)

Same high-level `deepcor.DeepCor` API as the v2 notebook — the **only**
difference is `model_version="v1"`. The original CVAE has no confound
conditioning, so `confounds` is optional and ignored here.

In [2]:
# GPU check (optional)
deepcor.utils.check_gpu_and_speedup(tensor_size=(1000, 1000), n_iter=100)

{'gpu_available': True,
 'gpu_name': 'Tesla V100-SXM2-16GB',
 'cpu_time': 0.02009997844696045,
 'gpu_time': 0.0001961398124694824,
 'speedup': 102.47781005749572}

In [ ]:
# Cell tagged parameters
s=0
r=1

param_epochs=5
param_repetitions=5

analysis_name='test-simple-V1'

In [3]:
# ---- Data paths (the only thing you normally edit) ----
bids_path = "../Data/fMRI-Data/studyforrest-fmriprep/"

subs = sorted(d for d in os.listdir(bids_path) if d.startswith("sub-"))
sub_id, run = subs[s], str(r)

session, task = "ses-localizer", "objectcategories"
space = "MNI152NLin2009cAsym"
base = os.path.join(bids_path, sub_id, session, "func")

epi_path = os.path.join(
    base, f"{sub_id}_{session}_task-{task}_run-{run}_bold_space-{space}_preproc.nii.gz"
)
gm_mask_path = os.path.join(bids_path, "mask_roi.nii")
cf_mask_path = os.path.join(bids_path, "mask_roni.nii")

output_dir = os.path.join(
    "../Data/DeepCor-Outputs", analysis_name, f"S{s}-R{r}-cvae_v1"
)

for p in (epi_path, gm_mask_path, cf_mask_path):
    assert os.path.exists(p), f"missing: {p}"
print("EPI:", epi_path)
print("output_dir:", output_dir)

EPI: ../Data/fMRI-Data/studyforrest-fmriprep/sub-01/ses-localizer/func/sub-01_ses-localizer_task-objectcategories_run-4_bold_space-MNI152NLin2009cAsym_preproc.nii.gz
output_dir: ../Data/DeepCor-Outputs/forrest-simple-v1-jupyter/S0-R4-cvae_v1


In [4]:
# ---- Train + denoise ----
# Config is optional; omit it for sensible defaults. v1 ignores confounds.
#
# No `dashboard`/`on_epoch` => nothing is plotted interactively; the training
# dashboard is still saved to output_dir every epoch. A per-epoch progress
# line (subject/run, started, elapsed, ETA) is printed below.
denoiser = deepcor.DeepCor(model_version="v1")
denoiser.config.training.n_epochs = param_epochs
denoiser.config.training.n_repetitions = param_repetitions

result = denoiser.fit_denoise(
    epi_path,
    gm_mask_path,
    cf_mask_path,
    confounds=None,      # v1 has no confound conditioning
    output_dir=output_dir,
    subject_idx=s,
    run_idx=r,
)

device is cuda:0
model_version is v1
Preparing training data...
obs_list.shape: (41147, 156)
noi_list.shape: (9105, 156)
upsampling noi_list_coords
obs_list.shape: (41147, 156)
noi_list.shape: (41147, 156)
S0R4 Training started at: 2026-06-17 13:00:39, elapsed: 0:00:31.3 Ens:1/5, E:5/5 ETA:0:02:05.2
S0R4 Training started at: 2026-06-17 13:00:39, elapsed: 0:01:04.2 Ens:2/5, E:5/5 ETA:0:01:36.4
S0R4 Training started at: 2026-06-17 13:00:39, elapsed: 0:01:37.3 Ens:3/5, E:5/5 ETA:0:01:04.9
S0R4 Training started at: 2026-06-17 13:00:39, elapsed: 0:02:10.4 Ens:4/5, E:5/5 ETA:0:00:32.6
S0R4 Training started at: 2026-06-17 13:00:39, elapsed: 0:02:43.4 Ens:5/5, E:5/5 ETA:0:00:00.5
Averaging ensemble predictions...


100%|███████████████████████████████████████████████████████████████| 5/5 [00:05<00:00,  1.16s/it]


signals averaged: 5
Saving comparison outputs (preproc, CompCor)...
Denoising complete! Output: ../Data/DeepCor-Outputs/forrest-simple-v1-jupyter/S0-R4-cvae_v1/denoised_deepcor.nii.gz


In [5]:
print("Denoised:", result.denoised_path)
print("Preproc :", result.preproc_path)
print("CompCor :", result.compcor_path)
result.denoised_path

Denoised: ../Data/DeepCor-Outputs/forrest-simple-v1-jupyter/S0-R4-cvae_v1/denoised_deepcor.nii.gz
Preproc : ../Data/DeepCor-Outputs/forrest-simple-v1-jupyter/S0-R4-cvae_v1/preproc.nii.gz
CompCor : ../Data/DeepCor-Outputs/forrest-simple-v1-jupyter/S0-R4-cvae_v1/denoised_compcor.nii.gz


'../Data/DeepCor-Outputs/forrest-simple-v1-jupyter/S0-R4-cvae_v1/denoised_deepcor.nii.gz'